In [ ]:
# === RUNTIME RESTORATION & ENVIRONMENT SETUP ===
# Restore project files if starting in a fresh Colab session
import os, sys, shutil, zipfile

# 1. Extract handoff archive if present in /content
zip_path = "/content/gemma_paper_handoff.zip"
if os.path.exists(zip_path):
    print(f"Unpacking {zip_path} into /content...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("/content")
    print("Project restored successfully.")

# 2. Ensure /content is on python path
if "/content" not in sys.path:
    sys.path.insert(0, "/content")

# 3. Install compatible dependencies
!pip install -q "bitsandbytes>=0.50.0" accelerate datasets

# 4. Verify GPU availability
import torch
cuda_avail = torch.cuda.is_available()
print("=" * 60)
print(f"CUDA Acceleration Active : {cuda_avail}")
if cuda_avail:
    print(f"Primary GPU Device       : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Capacity            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: GPU is not attached! 8-bit model loading requires a CUDA GPU.")
    print("Please select: Runtime -> Change runtime type -> T4 GPU")
print("=" * 60)


# Reproducible Pilot: Gemma 4 Code Representation & Issue Repair

**Research Question:**
> *For one frozen coding model, how does the representation of retrieved repository code affect test-verified issue repair under a fixed evidence-token budget?*

### Pilot Experimental Configuration
- **Model:** `google/gemma-4-E4B-it` (Google DeepMind multimodal 8B/effective 4B PLE architecture)
- **Quantization:** `bitsandbytes` 8-bit (`BitsAndBytesConfig(load_in_8bit=True)`). *If VRAM is constrained, 4-bit NF4 configuration is provided.*
- **Target Runtime:** Google Colab (Tesla T4 GPU with ~15GB VRAM or any runtime GPU)
- **Training:** Frozen model (Zero fine-tuning, no LoRA)
- **Evidence Ceiling:** 4,000 tokens
- **Max Generated Tokens:** 2,048 tokens
- **Batch Size:** 1
- **Thinking Mode:** Explicitly disabled (`enable_thinking=False`) for this baseline pilot
- **Benchmark:** 3 SWE-bench Lite development tasks
- **Evaluation Environment:** WSL / Linux with containerized Docker test runner

---

## 1. Permanent Storage & Google Drive Mount
Colab runtimes are ephemeral. To ensure notebooks, logs, and environment manifests persist across sessions, we mount Google Drive and create `gemma_paper`.

In [ ]:
import os
import shutil

# Define permanent project workspace paths
LOCAL_WORKSPACE = "/content/gemma_paper"
DRIVE_WORKSPACE = "/content/drive/MyDrive/gemma_paper"

os.makedirs(LOCAL_WORKSPACE, exist_ok=True)

try:
    from google.colab import drive
    print("Mounting Google Drive for persistent storage...")
    drive.mount("/content/drive", force_remount=False)
    os.makedirs(DRIVE_WORKSPACE, exist_ok=True)
    print(f"Permanent Google Drive workspace active: {DRIVE_WORKSPACE}")
except Exception as e:
    print(f"Drive mount skipped or unavailable ({e}). Files will reside in {LOCAL_WORKSPACE}.")

## 2. Dependency Verification & Installation
We install `bitsandbytes` while **preserving Colab's pre-installed PyTorch** and CUDA toolkits without unintended downgrades.

In [ ]:
# Install bitsandbytes without touching or reinstalling PyTorch
!pip install -q "bitsandbytes>=0.50.0"

import sys
sys.path.insert(0, "/content")

import torch
import transformers
import bitsandbytes as bnb
import accelerate

print("Dependency Versions Loaded:")
print(f"  PyTorch       : {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()} (Version: {torch.version.cuda})")
print(f"  Transformers  : {transformers.__version__}")
print(f"  BitsAndBytes  : {bnb.__version__}")
print(f"  Accelerate    : {accelerate.__version__}")

Dependency Versions Loaded:
  PyTorch       : 2.11.0+cu130
  CUDA available: True (Version: 13.0)
  Transformers  : 5.22.0
  BitsAndBytes  : 0.50.2
  Accelerate    : 1.15.0


## 3. Hardware & Runtime Diagnostics Probe
Inspect GPU model, VRAM, RAM, disk space, and verify floating-point capability.
> **Note on Tesla T4:** Tesla T4 (Turing, compute capability 7.5) does not possess native bfloat16 tensor cores. For unquantized modules, we explicitly use `torch.float16` to ensure hardware acceleration and prevent software emulation overhead.

In [ ]:
from gemma_paper.env_probe import print_diagnostic_summary, get_environment_info

# Run the complete environment probe
env_info = get_environment_info()
print_diagnostic_summary()

# Verify hardware readiness
assert env_info["cuda"]["is_available"], "FATAL: GPU acceleration is required for 8-bit BitsAndBytes quantization!"
print("Hardware readiness: PASS")

RUNTIME ENVIRONMENT DIAGNOSTIC REPORT
Python Version : 3.13.16
OS Platform    : Linux-6.6.159+-x86_64-with-glibc2.39

--- Hardware Resources ---
System RAM     : 9.52 GB free / 12.67 GB total (24.9% used)
Disk Space     : 112.27 GB free / 112.64 GB total (0.3% used)

--- GPU & CUDA Status ---
GPU Detected   : Tesla T4
Compute Cap    : 7.5
Total VRAM     : 14.56 GB
CUDA Version   : 13.0
Native BF16 TC : False
Selected Dtype : torch.float16 (Unquantized module precision)

--- Key Library Versions ---
  torch           : 2.11.0+cu130       [OK]
  torchvision     : 0.26.0+cu130       [OK]
  transformers    : 5.22.0             [OK]
  accelerate      : 1.15.0             [OK]
  bitsandbytes    : 0.50.2             [OK]
  huggingface_hub : 1.33.0             [OK]
Hardware readiness: PASS


## 4. Secure Hugging Face Credential Access
Retrieve `HF_TOKEN` from **Colab Secrets** (`google.colab.userdata`).
- If accessing gated resources or high rate limits, store `HF_TOKEN` in Colab's left sidebar (Key icon).
- Credentials are never printed or written to disk.

In [ ]:
from gemma_paper.model_loader import get_hf_token

hf_token = get_hf_token()
if hf_token:
    print("Hugging Face token detected securely from Colab Secrets / Environment. (Value hidden)")
else:
    print("No HF_TOKEN found in Secrets. Proceeding with anonymous access (standard rate limits apply).")

No HF_TOKEN found in Secrets. Proceeding with anonymous access (standard rate limits apply).


## 5. Model & Processor Loading
### Architecture Compatibility
`google/gemma-4-E4B-it` is a multimodal architecture in the Gemma 4 family. In Hugging Face `transformers`:
1. **Processor:** Must use `AutoProcessor` (maps to `Gemma4Processor`), which handles modality tokens and the canonical jinja chat template.
2. **Model Class:** `AutoModelForImageTextToText` (which resolves to `Gemma4ForConditionalGeneration`).
3. **Quantization:** `BitsAndBytesConfig(load_in_8bit=True)` with `torch_dtype=torch.float16` for unquantized modules on Tesla T4.

> **4-Bit Alternative:** If 8-bit exceeds available VRAM during intense workloads, pass `use_4bit=True` to configure NF4 quantization.

In [ ]:
from gemma_paper.model_loader import load_gemma4_and_processor

MODEL_ID = "google/gemma-4-E4B-it"
USE_4BIT = False  # Set to True only if 8-bit encounters VRAM exhaustion

# Load model and processor with 8-bit bitsandbytes
model, processor, unquantized_dtype = load_gemma4_and_processor(
    model_id=MODEL_ID,
    use_4bit=USE_4BIT
)

print(f"\nModel successfully loaded: {model.__class__.__name__}")
print(f"Processor successfully loaded: {processor.__class__.__name__}")

## 6. Quantization & Device Placement Verification
Verify that `bitsandbytes` quantized layers (`Linear8bitLt`) loaded into GPU memory, inspect unquantized components, and report VRAM utilization.

In [ ]:
from gemma_paper.model_loader import verify_quantization_and_placement

# Execute deep inspection of model parameters
quant_report = verify_quantization_and_placement(model)

## 7. Chat Template & Thinking Mode Configuration
Gemma 4 models include built-in reasoning tokens (`<|think|>`). For this initial pilot, **thinking mode is explicitly disabled** (`enable_thinking=False`) to establish a clean direct-inference baseline.

In [ ]:
from gemma_paper.generation_smoke import format_chat_input

sample_prompt = "Write a hello world Python function."

# Check chat template rendering with enable_thinking=False
sample_inputs = format_chat_input(processor, sample_prompt, enable_thinking=False)
rendered_text = processor.decode(sample_inputs["input_ids"][0], skip_special_tokens=False)

print("Rendered Input String (enable_thinking=False):")
print(repr(rendered_text))
assert "<|think|>" not in rendered_text, "Thinking token should not be present when enable_thinking=False"

## 8. Generation Smoke Tests
> **IMPORTANT NOTICE:** These outputs are **smoke tests** designed solely to verify token generation pipeline integrity, memory bounds, and latency. They are **not** evaluation results for issue repair.

In [ ]:
from gemma_paper.generation_smoke import (
    run_smoke_test,
    SHORT_SANITY_PROMPT,
    SWE_BENCH_REPRESENTATIVE_CODE_PROMPT
)

smoke_results = []

# Smoke Test 1: Short Sanity Check
smoke_1 = run_smoke_test(
    model=model,
    processor=processor,
    prompt=SHORT_SANITY_PROMPT,
    test_label="Smoke Test 1: Short Sanity Prompt",
    max_new_tokens=64,
    enable_thinking=False,
    temperature=0.0
)
smoke_results.append(smoke_1)

# Smoke Test 2: SWE-bench Representative Code Prompt with Longer Output Allowance
smoke_2 = run_smoke_test(
    model=model,
    processor=processor,
    prompt=SWE_BENCH_REPRESENTATIVE_CODE_PROMPT,
    test_label="Smoke Test 2: SWE-bench Representative Code Prompt",
    max_new_tokens=1024,  # Within the 2,048 max generated tokens allowance
    enable_thinking=False,
    temperature=0.0
)
smoke_results.append(smoke_2)

## 9. Save Environment & Experiment Manifest
We record an immutable JSON manifest capturing model revision, template hash, dependency versions, hardware properties, and generation configurations to guarantee reproducibility.

In [ ]:
from gemma_paper.manifest import build_manifest, save_manifest

manifest = build_manifest(
    model=model,
    processor=processor,
    model_id=MODEL_ID,
    quantization_type="bitsandbytes_4bit" if USE_4BIT else "bitsandbytes_8bit",
    unquantized_dtype=str(unquantized_dtype),
    enable_thinking=False,
    smoke_test_results=smoke_results
)

# Save locally and mirror to Google Drive
saved_manifest_path = save_manifest(manifest, output_dir=LOCAL_WORKSPACE)

# Also export notebook copy to permanent Google Drive workspace
if os.path.exists(DRIVE_WORKSPACE):
    shutil.copy2("/content/gemma_paper/gemma_evidence_pilot.ipynb", os.path.join(DRIVE_WORKSPACE, "gemma_evidence_pilot.ipynb"))
    print(f"Notebook mirrored to Google Drive: {os.path.join(DRIVE_WORKSPACE, 'gemma_evidence_pilot.ipynb')}")

## 10. Failure Triage & Diagnostic Guide

| Failure Symptom | Underlying Cause | Smallest Prescribed Correction |
| :--- | :--- | :--- |
| `CUDA out of memory` during model load | Model weights and KV cache exceed 15GB VRAM | Switch `USE_4BIT = True` in Cell 5. Do not silently offload layers to CPU. |
| `ImportError: cannot import name 'Gemma4ForConditionalGeneration'` | Outdated `transformers` version | Ensure `transformers >= 4.49.0` (Colab currently installs 5.x). |
| `ValueError: bfloat16 is not supported` | Turing GPU (Tesla T4) executing BF16 ops | Ensure `torch_dtype=torch.float16` is used for unquantized modules. |
| Rate Limit / 401 / 403 on HF Hub | Anonymous requests restricted | Add `HF_TOKEN` in Colab Secrets (Key icon on left sidebar). |
| `RuntimeError: No bitsandbytes quantized layers found` | Quantization failed to wrap linear modules | Verify `bitsandbytes` installation matches PyTorch CUDA version. |

# Stage 2: Task Preparation & Evidence Representation Formats

## 11. Deterministic Task Selection & Repository Checkout
- **Pinned dataset revision:**  (),  split.
- **Selection rule:** Lexicographical repository ordering, first 3 distinct repos (, , ), lowest instance ID per repo.
- **Solver input sanitization:** Reference patches, test patches, evaluator labels, and hints are strictly excluded.

In [ ]:
from gemma_paper.dataset_prep import load_and_select_tasks, save_task_manifest, prepare_repository_at_base_commit

selected_tasks, manifest_meta = load_and_select_tasks()
save_task_manifest(manifest_meta, "/content/task_manifest.json")

print("
Deterministic Task Manifest:")
for t in manifest_meta["selected_tasks"]:
    print(f"  - {t['instance_id']:<36} | Repo: {t['repo']:<28} | Base: {t['base_commit'][:10]}")

## 12. Shared Lexical Retrieval Ranking & Code Entity Extraction
- Extracts module headers, top-level statements (imports, constants), functions, and classes via Python .
- Computes deterministic BM25 lexical ranking against the issue problem statement.
- Documented tie-breaking: .
- Caches one shared ranking per task to ensure identical ordering across all conditions.

In [ ]:
import os
from gemma_paper.retrieval import build_and_cache_task_ranking

task_rankings = {}
REPOS_DIR = "/content/task_repos"

for task_record in selected_tasks:
    task_id = task_record["instance_id"]
    repo_name = task_record["repo"]
    base_commit = task_record["base_commit"]
    prob_statement = task_record["problem_statement"]

    repo_dir = os.path.join(REPOS_DIR, task_id.replace("/", "__"))
    prepare_repository_at_base_commit(repo_name, base_commit, repo_dir)

    ranking = build_and_cache_task_ranking(
        task_id=task_id,
        repo_dir=repo_dir,
        problem_statement=prob_statement
    )
    task_rankings[task_id] = ranking
    print(f"[{task_id}] Indexed {ranking['indexed_file_count']} files, ranked {ranking['ranked_entities_count']} entities.")

## 13. Evidence Representation Packing (W, A, B, C)
Pack evidence under the strict 4,000 token budget ceiling measured with the model tokenizer ():
- **W (Whole source files):** Complete source files in retrieval rank order; oversized files are skipped without silent truncation.
- **A (Complete code-entity bodies):** Full bodies of top-ranked functions, classes, and module headers.
- **B (Exact signatures, bounded docstrings, selected excerpts):** Exact definition signatures, bounded docstrings (<= 8 lines), and verbatim excerpts.
- **C (Full top-ranked + compact supporting):** Rank 1 entity rendered in full (Condition A); Rank 2+ supporting entities rendered compact (Condition B).
- **Fidelity & Deduplication:** Original source slices (no AST unparsing), explicit omission markers, and span deduplication.

In [ ]:
from transformers import AutoProcessor
from gemma_paper.packet_runner import generate_all_12_evidence_packets, print_packet_summary_table

try:
    processor
except NameError:
    processor = AutoProcessor.from_pretrained("google/gemma-4-E4B-it")

all_packets, summary_rows = generate_all_12_evidence_packets(processor=processor)
print_packet_summary_table(summary_rows)

## 14. Verification Suite & Quality Assurance Audit
Automated checks execute across all 12 packets:
1. **Source Fidelity:** Asserts every extracted line slice matches the exact bytes in the repository at .
2. **Budget Enforcement:** Asserts evidence tokens <= 4,000 tokens for every packet.
3. **Forbidden-Field Leakage:** Confirms zero test patches, gold patches, or issue hints appear in solver packets.
4. **Determinism:** Bitwise reproducibility audit.

In [ ]:
import json

total_packets = len(all_packets)
passed_packets = sum(1 for p in all_packets if p["qa_verification"]["all_passed"])

print(f"Total Packets Generated: {total_packets}")
print(f"Packets Passing All Verification Checks: {passed_packets} / {total_packets}")
assert passed_packets == total_packets, "QA check failed"
print("All 12 evidence packets verified: Source fidelity, budget ceiling, and leakage checks PASSED.")

## 15. Sample Evidence Packet Inspection
Inspect a representative evidence packet ( under **Condition C**) to review headers, source ranges, and omission markers.

In [ ]:
sample_path = "/content/evidence_packets/marshmallow-code__marshmallow-1343__C.json"
with open(sample_path, "r", encoding="utf-8") as f:
    sample_packet = json.load(f)

print(f"Packet ID              : {sample_packet['packet_id']}")
print(f"Task ID                : {sample_packet['task_id']}")
print(f"Condition              : {sample_packet['condition']}")
print(f"Evidence Tokens        : {sample_packet['actual_evidence_tokens']} / 4000 (Unused: {sample_packet['unused_budget_tokens']})")
print(f"Total Chat Prompt Toks : {sample_packet['total_chat_prompt_tokens']}")
print(f"Selected Spans Count   : {len(sample_packet['selected_spans'])}")
print("-" * 65)
print("EVIDENCE PREVIEW (first 1,200 chars):")
print(sample_packet["evidence_text"][:1200])
print("
... [rest of evidence]")

## 16. Independent Evaluator Setup & Preflight Procedure (WSL/Linux + Docker)
Evaluation requires Docker and runs independently in WSL2 or native Linux using official SWE-bench harness pinned at **v5.0.1**.

### Preflight Protocol:
1. **Baseline Empty Patch Check:** Verifies  tests fail on unmodified repository.
2. **Reference Gold Control Check:** Verifies ground-truth patches resolve tests.
> **Isolation Protocol:** Reference patches and test patch definitions are stored strictly within  and are never exposed to solver inputs or evidence packets.

In [ ]:
import sys
sys.path.insert(0, "/content")
from evaluator.preflight import generate_preflight_prediction_files, check_preflight_results

# Generate isolated preflight prediction files
empty_path, gold_path = generate_preflight_prediction_files()

# Audit preflight execution status
status = check_preflight_results("/content/logs/evaluation")
print(f"
Evaluator Status in Colab: {status['preflight_status']}")
print(f"Note: {status.get('message', '')}")

## 17. Prediction Importer & Report Parser
- **Importer:** Extracts unified diffs, validates syntax, and constructs unique run IDs to avoid SWE-bench cache collisions.
- **Parser:** Traverses , separates 5 distinct outcome states (, , , , ), and joins with experimental condition metadata.

In [ ]:
from evaluator.prediction_importer import generate_unique_run_id, validate_patch_syntax
from evaluator.report_parser import join_evaluator_report_to_configuration

# Demonstrate unique collision-free run ID generation
sample_run_id = generate_unique_run_id(model_tag="gemma4_E4B_it", condition="C", budget=4000, attempt=1)
print(f"Generated Unique Run ID: {sample_run_id}")

# Verify patch validator on clean vs corrupted diff
is_valid, reason = validate_patch_syntax("diff --git a/test.py b/test.py
--- a/test.py
+++ b/test.py
@@ -1,3 +1,3 @@
-x = 1
+x = 2
")
print(f"Validator Check: valid={is_valid}, status={reason}")

# Stage 3: Resumable Repair Inference & SWE-bench Prediction Export

### Experimental Protocol & Invariant Constraints:
1. **Frozen Baseline:** Zero fine-tuning, no LoRA. Single model checkpoint (`google/gemma-4-E4B-it`) in 8-bit quantization (`BitsAndBytesConfig(load_in_8bit=True)`).
2. **Fixed Decoding Settings:**
   - Strategy: Greedy decoding (`do_sample=False`, `temperature=0.0`)
   - Thinking Mode: Explicitly disabled (`enable_thinking=False`)
   - Max Generated Tokens: `2,048` tokens
   - Attempts: Exactly one attempt per task and condition (`attempt=1`)
   - *Note on Hardware Reproducibility:* Greedy decoding is deterministic on a fixed GPU architecture, but cross-hardware floating-point non-associativity means bitwise reproducibility across different GPU architectures cannot be claimed.
3. **Structured Patch Output Format:**
   - Strict `<<<<<<< SEARCH ... ======= ... >>>>>>> REPLACE` blocks.
   - Edits must have exact, unique matches in the repository file at `base_commit`.
   - Rejects invalid/ambiguous edits deterministically rather than applying fuzzy heuristics or recursive LLM repair.
4. **Information Isolation:**
   - Solver receives only the issue problem statement and the 4,000-token evidence packet.
   - Gold reference patches and evaluator test patches are strictly quarantined in `/content/evaluator/` and never leaked into solver context.
5. **Per-Attempt Checkpointing & Resumability:**
   - Checkpointed immediately after every attempt to `/content/inference_output/checkpoints/`.
   - Resume key incorporates `task_id`, `condition`, `budget`, `prompt_hash`, `config_hash`, and `attempt_number`.
   - If Colab disconnects or restarts, re-running the cell instantly skips completed attempts.
6. **SWE-bench Predictions & Artifact Bundling:**
   - Official prediction records exported per condition (`predictions__W.jsonl`, `predictions__A.jsonl`, etc.) and combined (`predictions__all_12.jsonl`).
   - Packaged into `/content/inference_output/gemma_repair_inference_bundle.tar.gz` and mirrored to Google Drive.
   - Verdict initialized to `PENDING_INDEPENDENT_EVALUATION`.

## 18. Inference Environment Verification & Preflight Audit
Verifies evaluator status, checks configuration hash invariance, and confirms model readiness.

In [ ]:
import os
import sys
sys.path.insert(0, "/content")

from evaluator.preflight import check_preflight_results
from gemma_paper.repair_runner import (
    get_configuration_hash,
    CONDITIONS,
    EVIDENCE_BUDGET,
    MAX_OUTPUT_TOKENS
)
from gemma_paper.model_loader import load_gemma4_and_processor

# 1. Verify Evaluator Preflight Status
eval_status = check_preflight_results("/content/logs/evaluation")
print(f"Independent Evaluator Preflight Status: {eval_status['preflight_status']}")
print(f"Evaluator Guidance: {eval_status.get('message', 'Ready for inference.')}")

# 2. Check and load model if not already present in memory
MODEL_ID = "google/gemma-4-E4B-it"
if 'model' not in globals() or 'processor' not in globals():
    print(f"\nLoading {MODEL_ID} in 8-bit...")
    model, processor, unquantized_dtype = load_gemma4_and_processor(MODEL_ID, use_4bit=False)
else:
    print(f"\nReusing loaded model ({model.__class__.__name__}) and processor from session.")

# 3. Verify Invariant Configuration Hash
config_hash = get_configuration_hash()
print(f"Frozen Configuration SHA-256 Hash: {config_hash}")
print(f"Target Conditions: {CONDITIONS}")
print(f"Evidence Budget: {EVIDENCE_BUDGET} tokens | Max Output: {MAX_OUTPUT_TOKENS} tokens")
print(f"Decoding: Greedy (do_sample=False) | Thinking: Disabled (enable_thinking=False)")

## 19. Resumable Repair Inference Execution (12 Packets)
Executes greedy generation across all 12 evidence packets (3 tasks x 4 conditions: W, A, B, C).
Checkpoints are written after each individual attempt. Re-running this cell skips already completed checkpoints.

In [ ]:
from gemma_paper.repair_runner import run_resumable_pilot_inference

OUTPUT_DIR = "/content/inference_output"

print("Starting Resumable Repair Inference across 12 Evidence Packets...")
print("Checkpoints are saved immediately after every single attempt to prevent progress loss.\n")

attempt_records, exported_pred_files = run_resumable_pilot_inference(
    model=model,
    processor=processor,
    evidence_packets_dir="/content/evidence_packets",
    repos_base_dir="/content/task_repos",
    output_dir=OUTPUT_DIR
)

print(f"\nAll 12 attempts processed. Checkpoints directory: {OUTPUT_DIR}/checkpoints")

## 20. Resumable Batch Inference Execution (All 12 W, A, B, C Packets)
Executes greedy generation sequentially across all 12 W/A/B/C evidence packets.
Checkpoints are written after each individual attempt. Re-running skips already completed checkpoints.

In [ ]:
from gemma_paper.repair_runner import run_resumable_pilot_inference, print_inference_summary_table

OUTPUT_DIR = "/content/inference_output"

print("Starting Resumable Repair Inference across all 12 W/A/B/C Evidence Packets...")
print("Checkpoints are saved immediately after every single attempt to prevent progress loss.\n")

attempt_records, exported_pred_files = run_resumable_pilot_inference(
    model=model,
    processor=processor,
    evidence_packets_dir="/content/evidence_packets",
    repos_base_dir="/content/task_repos",
    output_dir=OUTPUT_DIR
)

# Render structured results table
print_inference_summary_table(attempt_records)

# Breakdown of conversion outcomes
success_count = sum(1 for r in attempt_records if r["conversion_status"] == "SUCCESS")
rejected_count = sum(1 for r in attempt_records if r["conversion_status"] != "SUCCESS")
trunc_count = sum(1 for r in attempt_records if r["finish_reason"] == "length_truncated")

print(f"\nRepair Extraction Statistics:")
print(f"  Valid Unified Diffs Synthesized : {success_count} / {len(attempt_records)}")
print(f"  Rejected Search/Replace Edits   : {rejected_count} / {len(attempt_records)}")
print(f"  Output Length Truncations       : {trunc_count} / {len(attempt_records)}")


## 21. SWE-bench Prediction Exports & Standalone Evaluation Package
Exports official SWE-bench prediction JSONL files per condition and generates `gemma_repair_inference_bundle.tar.gz`.

In [ ]:
import json
import os

print("Exported SWE-bench Prediction Files:")
for cond_key, file_path in exported_pred_files.items():
    if os.path.exists(file_path):
        size_kb = os.path.getsize(file_path) / 1024
        with open(file_path, "r", encoding="utf-8") as f:
            lines = [json.loads(line) for line in f]
        non_empty = sum(1 for entry in lines if entry.get("model_patch", "").strip())
        print(f"  [{cond_key}] {file_path} ({size_kb:.1f} KB, {len(lines)} instances, {non_empty} with valid patch)")

bundle_path = os.path.join(OUTPUT_DIR, "gemma_repair_inference_bundle.tar.gz")
print(f"\nStandalone Evaluation Archive: {bundle_path}")
print(f"Archive Size: {os.path.getsize(bundle_path) / 1024:.1f} KB")
print("\nNext Steps for Docker Evaluation (in WSL2 or Linux):")
print("1. Download `gemma_repair_inference_bundle.tar.gz` (or retrieve from Google Drive).")
print("2. Extract in environment: tar -xzf gemma_repair_inference_bundle.tar.gz")
print("3. Execute official SWE-bench evaluation harness: bash /content/evaluator/run_evaluator.sh")